In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, learning_curve


In [3]:
# ---------- Synthetic Titanic Dataset ----------
def make_synthetic_titanic(n=800, random_state=42):
    rng = np.random.RandomState(random_state)

    pclass = rng.choice([1, 2, 3], size=n, p=[0.24, 0.21, 0.55])
    sex = rng.choice(["male", "female"], size=n, p=[0.62, 0.38])
    age = np.clip(rng.normal(30, 14, size=n), 0.42, 80)
    fare = np.clip(rng.lognormal(mean=3.2, sigma=0.8, size=n), 3, 512)
    sibsp = rng.poisson(0.5, size=n)
    parch = rng.poisson(0.3, size=n)
    embarked = rng.choice(["S", "C", "Q"], size=n, p=[0.72, 0.19, 0.09])

    prob = (
        0.30
        + 0.28 * (sex == "female")
        + 0.18 * (pclass == 1)
        - 0.10 * (pclass == 3)
        + 0.15 * (age < 16)
        - 0.04 * sibsp
        - 0.03 * parch
    )
    prob = np.clip(prob, 0.01, 0.99)

    survived = (rng.rand(n) < prob).astype(int)

    df = pd.DataFrame({
        "Pclass": pclass,
        "Sex": sex,
        "Age": age,
        "SibSp": sibsp,
        "Parch": parch,
        "Fare": fare,
        "Embarked": embarked,
        "Survived": survived
    })

    df.loc[rng.rand(n) < 0.15, "Age"] = np.nan
    df.loc[rng.rand(n) < 0.02, "Embarked"] = np.nan

    return df


In [4]:
# ---------- Load Dataset ----------
def get_data():
    csv_path = "titanic.csv"
    if os.path.exists(csv_path):
        return pd.read_csv(csv_path), "titanic.csv"
    else:
        df = make_synthetic_titanic()
        df.to_csv(csv_path, index=False)
        return df, "synthetic -> titanic.csv"

In [5]:
# ---------- Main ----------
def main():
    df, source = get_data()
    print(f"Dataset source: {source}")

    features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
    X = df[features]
    y = df["Survived"].astype(int)

    cat_cols = ["Sex", "Embarked", "Pclass"]
    num_cols = [c for c in X.columns if c not in cat_cols]

    preprocess = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ])

    models = {
        "Logistic Regression": LogisticRegression(max_iter=2000),
        "Decision Tree": DecisionTreeClassifier(random_state=42)
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    print("\n10-Fold Cross-Validation Accuracy")
    for name, model in models.items():
        pipe = Pipeline([("prep", preprocess), ("model", model)])
        scores = cross_val_score(pipe, X, y, cv=cv, scoring="accuracy")
        print(f"{name}: {scores.mean():.3f} ± {scores.std():.3f}")

    def plot_learning_curve(model, title, filename):
        pipe = Pipeline([("prep", preprocess), ("model", model)])
        train_sizes, train_scores, val_scores = learning_curve(
            pipe, X, y, cv=cv, scoring="accuracy",
            train_sizes=np.linspace(0.1, 1.0, 5),
            shuffle=True, random_state=42
        )

        plt.figure()
        plt.plot(train_sizes, train_scores.mean(axis=1), label="Training Accuracy")
        plt.plot(train_sizes, val_scores.mean(axis=1), label="Validation Accuracy")
        plt.fill_between(train_sizes,
                         train_scores.mean(axis=1) - train_scores.std(axis=1),
                         train_scores.mean(axis=1) + train_scores.std(axis=1),
                         alpha=0.2)
        plt.fill_between(train_sizes,
                         val_scores.mean(axis=1) - val_scores.std(axis=1),
                         val_scores.mean(axis=1) + val_scores.std(axis=1),
                         alpha=0.2)
        plt.xlabel("Training Samples")
        plt.ylabel("Accuracy")
        plt.title(title)
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(filename, dpi=150)
        plt.close()

    plot_learning_curve(models["Logistic Regression"],
                        "Learning Curve: Logistic Regression (Titanic)",
                        "learning_curve_logreg.png")

    plot_learning_curve(models["Decision Tree"],
                        "Learning Curve: Decision Tree (Titanic)",
                        "learning_curve_dtree.png")


if __name__ == "__main__":
    main()

Dataset source: titanic.csv

10-Fold Cross-Validation Accuracy
Logistic Regression: 0.682 ± 0.028
Decision Tree: 0.604 ± 0.058
